In [1]:
import pandas as pd
data_train = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
data_valid = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")

In [2]:
import pandas as pd
import numpy as np
import torch
import gc
from torch_geometric.data import Data # thư viên Data giúp lưu trữ đồ thị lưới 
from torch_geometric.utils import coalesce
from tqdm.notebook import tqdm
import os

# bật cảnh báo trùng lặp của KMP (Intel MKL) để tránh lỗi crash khi dùng nhiều luồng trên CPU
os.environ["KMP_DUPLICATE_BLOCK_WARNING"] = "TRUE"

# xây dựng đồ thị Flow-based Graph có đặc trưng cạnh
def build_ip_topology_graph(df, train_mask, valid_mask, test_mask, window_size=10):
    print(f"Bước 1: Trích xuất Node Features (X) và Label (y)...")
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    
    # các cột cần drop 
    cols_present = [c for c in cols_to_drop if c in df.columns]

    # các cột còn lại sẽ là feature
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    # tạo một numpy array chứa các feature
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    
    # ascontiguousarray: đảm bảo dữ liệu được lưu trữ liên tục trong bộ nhớ
    features_np = np.ascontiguousarray(features_np)
    
    # lấy node features (x_tensor) và label (y_tensor)
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()



    print(f"Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...")

    # lấy các giá trị để xây dựng cạnh
    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    # tạo một mảng split_id để đánh dấu tập Train (0), Valid (1), Test (2)
    split_id = np.zeros(len(df), dtype=np.int8)
    split_id[valid_mask.numpy()] = 1
    split_id[test_mask.numpy()] = 2
    

    # gom nhóm các gói tin theo ip để nối cạnh
    ip_to_indices = {}
    for idx in tqdm(range(len(df)), desc="[1/2] Gom nhóm Gói tin theo IP"):
        s_ip = src_ips[idx]
        d_ip = dst_ips[idx]
        
        # ip_to_indices: dictionary mapping mỗi IP (cả src và dst) đến danh sách các chỉ số của gói tin liên quan đến IP đó
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx:
            ip_to_indices[s_ip].append(idx)
            
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx:
            ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    
    # nối cạnh giữa các gói tin có cùng IP (src hoặc dst) trong một cửa sổ thời gian nhất định (window_size)
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Căng lưới đồ thị (Window = {window_size})"):
        n_flows = len(indices)
        if n_flows < 2: continue
        
        for i in range(1, n_flows):
            curr_idx = indices[i]
            start_w = max(0, i - window_size) # start_w: chỉ số bắt đầu của cửa sổ (window) để tìm các gói tin trước đó có cùng IP
            
            # duyệt qua các gói tin trước đó trong cửa sổ để tạo cạnh
            for j in range(start_w, i):
                past_idx = indices[j]
                
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx]) # tính khoảng thời gian giữa 2 gói tin
                
                # tuy nhiên, nếu 2 gói tin cách nhau hơn 60s thì không nói cạnh 
                if dt_raw > 60.0:
                    continue
                
                # tính toán các đặc trưng cạnh
                # chênh lệch thời gian: (Log scale + Min-Max Scaling về dải [0, 1])
                # giá trị max lý thuyết khi dt_raw = 60s là log1p(60,000,000) ~ 17.9 => chia cho 18
                dt = np.log1p(dt_raw * 1e6) / 18.0
                # binary liệu 2 gói tin này có chung địa chỉ IP nguồn không (cùng người gửi với nhau)
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                # liệu 2 gói tin này có cùng cổng nguồn không?
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                # Liệu 2 gói tin này có cùng cổng đích không?
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0

                # list lưu các đặc trưng cạnh
                attr = [dt, same_dir, same_sport, same_dport]
                
                # nối cạnh tuân thủ quy tắc thời gian
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr) 
                    
    print("\nHợp nhất và tạo Cạnh (Culling Edge Index)...")
    
    # Chuyển thẳng sang Tensor để giảm RAM overhead thay vì dùng pd.DataFrame
    edge_index = torch.tensor([all_src, all_dst], dtype=torch.long)
    edge_attr = torch.tensor(all_edge_attrs, dtype=torch.float32)
    
    # Giải phóng liền các list khổng lồ
    del all_src, all_dst, all_edge_attrs
    gc.collect()
    
    # Loại bỏ cạnh log trùng lặp bằng torch_geometric.utils.coalesce
    edge_index, edge_attr = coalesce(edge_index, edge_attr, reduce='mean')
    
    print(f"HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!")
    print(f"   - Số Nodes : {x_tensor.size(0):,}")
    print(f"   - Số Edges : {edge_index.size(1):,}")
    print(f"   - Số Features/Cạnh : {edge_attr.shape[1]}")
    
    del features_np
    gc.collect()
    
    # tạo đồ thị PyG Data với node features, edge index, edge attributes và label
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    
    # gắn mask cho tập train, valid, test vào đồ thị
    graph.train_mask = train_mask
    graph.valid_mask = valid_mask
    graph.test_mask = test_mask
    return graph

In [3]:
import gc
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from torch_geometric.loader import NeighborLoader
import torch
import numpy as np
import pandas as pd

print("=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===")

# tạo 1 cột để đánh dấu tập dữ liệu (train/valid/test) trước khi gộp để tránh Data Leakage
print("Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...")
data_train['split_tag'] = 0
data_valid['split_tag'] = 1
data_test['split_tag'] = 2

# concat lại thành một dataframe chuẩn
df_all = pd.concat([data_train, data_valid, data_test], ignore_index=True)

# xóa các dataframe gốc để giải phóng bộ nhớ
del data_train, data_valid, data_test
gc.collect()

# chuyển datetime sang số để sắp xếp và giữ lại micro giây
df_all['timestamp_num'] = pd.to_datetime(df_all['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

print("Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...")
# Sắp xếp tĩnh trực tiếp (INPLACE) để chặn Pandas tạo bản copy 9GB trong bộ nhớ RAM
df_all.sort_values(by='timestamp_num', inplace=True)
df_all.reset_index(drop=True, inplace=True)
gc.collect()

total_nodes = len(df_all)

# tạo mask cho đồ thị
print("Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...")
train_mask = torch.tensor((df_all['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((df_all['split_tag'] == 1).values, dtype=torch.bool)
test_mask  = torch.tensor((df_all['split_tag'] == 2).values, dtype=torch.bool)

# xóa cột split_tag
df_all.drop(columns=['split_tag'], inplace=True)
gc.collect()

# in ra thông tin thống kê về số lượng node và phân bố mask
print(f" - Tổng Node: {total_nodes:,}")
print(f" - Train Mask : {train_mask.sum().item():,} ({train_mask.float().mean()*100:.1f}%)")
print(f" - Valid Mask : {valid_mask.sum().item():,} ({valid_mask.float().mean()*100:.1f}%)")
print(f" - Test Mask  : {test_mask.sum().item():,} ({test_mask.float().mean()*100:.1f}%)")

print("Hoàn tất Data Prep cho Masks và Sorting Thời gian.")

# build đồ thị với đặc trưng cạnh và gắn mask
print("\n=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===")
super_graph_faiss = build_ip_topology_graph(df_all, train_mask, valid_mask, test_mask, window_size=10)
print("\nĐồ thị tổng IP Topo Đã Gắn Masks thành công!")

# tạo neighborloader (tương tự như dataloaders)
# lưu ý: neighborloader này sẽ không trả về 2048 sub-graph mà nó sẽ trả về một đồ thị kết hợp 2028 node đó và hàng xom hop 1, hop 2. 2 node cũng có thể có chung 1 node hàng xóm
# về lý thuyết, đồ thị này có nhiều nhất: 2048 + 2048*10 + 2048*10*5 node nếu không có node nào trùng hàng xóm
# khi đưa đồ thị này vào GAT thì nó sẽ trả lại embedding cho TẤT CẢ các node trong đồ thị kết hợp này, ta chỉ lấy 2048 embedding đầu tiên là 2028 embedding của node gốc
print("\nSet up IP Graph NeighborLoaders...")
train_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
test_loader_faiss  = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.test_mask,  batch_size=2048, shuffle=False, num_workers=0)

del df_all 
gc.collect()
print("Hoàn tất Data Prep cho IP Graph")

=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===
Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...
Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...
Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...
 - Tổng Node: 10,983,170
 - Train Mask : 3,294,945 (30.0%)
 - Valid Mask : 4,393,262 (40.0%)
 - Test Mask  : 3,294,963 (30.0%)
Hoàn tất Data Prep cho Masks và Sorting Thời gian.

=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/10983170 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/49 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 10,983,170
   - Số Edges : 169,023,865
   - Số Features/Cạnh : 4

Đồ thị tổng IP Topo Đã Gắn Masks thành công!

Set up IP Graph NeighborLoaders...
Hoàn tất Data Prep cho IP Graph


In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
import gc

# định nghĩa gat với 2 lớp
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    # Loại bỏ hoàn toàn dropout ở node features để giữ nguyên tín hiệu liên tục. Chỉ dùng edge_dropout nếu cần.
    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        # x = F.dropout(x, p=0.25, training=self.training) # Xóa node dropout
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        # x = F.dropout(x, p=0.25, training=self.training) # Xóa node dropout
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        # Thêm Dropout vừa phải TRƯỚC lớp Linear Classifier
        out = F.dropout(embedding, p=0.2, training=self.training)
        out = self.classifier(out)
        return out, embedding

def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=7):
    from sklearn.metrics import f1_score
    import numpy as np

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3) 
    
    # learning rate giảm một nửa nếu sau 2 epoch liên tiếp mà F1 validation không cải thiện 
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )
    
    # nếu có class_weights thì truyền vào để xử lý imbalanced dataset
    # CẤP CẬP: Ngừng ngay Label Smoothing (chuyển 0.1 về 0.0) vì nó đang phá nát các embedding của các lớp thiểu số
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.0)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.0)
        
    best_val_f1 = -1.0
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_nodes = 0
        all_train_preds = []
        all_train_labels = []
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
            batch_size = batch.batch_size
            
            # chỉ lấy 20248 embedding của 2028 node gốc để tính loss và đánh giá, bỏ qua hàng xóm hop 1, hop 2
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0) # dùng gradient clipping tránh exploding gradients
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            y_true = batch.y[:batch_size]
            
            all_train_preds.append(pred.cpu().numpy())
            all_train_labels.append(y_true.cpu().numpy())
            
            total_train_loss += loss.item() * batch_size
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
            
            del batch, out, emb, loss, pred, y_true
            
        avg_train_loss = total_train_loss / total_train_nodes
        train_f1 = f1_score(np.concatenate(all_train_labels), np.concatenate(all_train_preds), average='macro')
        
        pbar_train.close()
            
        model.eval()
        total_val_loss = 0
        total_val_nodes = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                
                out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                y_true = batch.y[:batch_size]
                
                all_val_preds.append(pred.cpu().numpy())
                all_val_labels.append(y_true.cpu().numpy())
                
                total_val_loss += loss.item() * batch_size
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}"})
                
                del batch, out, emb, loss, pred, y_true
                
        pbar_val.close()
                
        avg_val_loss = total_val_loss / total_val_nodes
        val_f1 = f1_score(np.concatenate(all_val_labels), np.concatenate(all_val_preds), average='macro')
        
        scheduler.step(val_f1)
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Macro F1: {train_f1:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Macro F1: {val_f1:.4f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate hiện tại: {current_lr:.6f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve_epochs = 0
            best_model_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Macro F1: {best_val_f1:.4f} | Val Loss: {avg_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện F1 {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\nĐã kích hoạt Early Stopping tại Epoch {epoch+1}! Lùi về checkpoint tốt nhất.")
                break
                
        # Gọi GC cuối mỗi epoch để dọn dẹp các mảng numpy từ sklearn/tqdm còn sót
        gc.collect()

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nHoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: {best_val_f1:.4f}) để dùng trích xuất Embedding!")
        
    return model

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm
import numpy as np
import torch
import gc
from torch_geometric.loader import NeighborLoader

# hàm trích xuất embedding từ model GAT
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings...")
    
    # loader dành cho lúc test/extract
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[30, 15], 
        input_nodes=mask,
        batch_size=512,
        shuffle=False,        
        num_workers=0 
    )
    
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        # truyền qua model để lấy embedding
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # lấy embedding của các node trong batch (chỉ lấy phần đầu tương ứng với batch_size vì NeighborLoader sẽ trả về cả node gốc và neighbor)
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        # xóa đi batch, emb để tránh nghẽn RAM
        del batch, emb, _
        
    pbar.close()
    
    # dọn rác cho các biến không dùng nữa
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # hợp final_embeddings thành ma trận (batch_size x embedding_dim) và final_labels thành vector (batch_size,)
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

# hàm huấn luyện và đánh giá XGBoost từ embedding của GAT (train bằng tập train, early stop bằng tập valid, test trên tập test)
def train_evaluate_xgboost(X_train_emb, y_train, X_valid_emb, y_valid, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    # tính toán class weights cho XGBoost dựa trên phân bố lớp tập train
    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    # Lấy power(w, 0.8) để cân bằng tốt hơn giữa smoothed và strict balanced
    weights_dict = {c: np.power(w, 0.8) for c, w in zip(unique_classes, raw_class_weights)}
    
    # Bỏ hoàn toàn các giới hạn (penalty/buff) hard-code thủ công do gây lệch Precision Class 11 và 10.
    # Ưu tiên thuật toán tự giải quyết không gian.

    # chuyển đổi thành mảng Sample Weight cho tập train
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    # train model xgboost
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,        # Tăng tốc độ học lên một chút để hội tụ tốt hơn
        max_depth=8,               # Giảm từ 10 xuống 8 để tránh chẻ quá sâu overfit vào một số lớp
        min_child_weight=2,        # Tăng lên 2 để yêu cầu node có đủ mẫu mới chia, giúp tăng Precision
        gamma=0.2,                 # Giảm chi phí phân nhánh
        reg_lambda=1.5,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.8,
        max_delta_step=2,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",  
        early_stopping_rounds=100  # Chờ kiên nhẫn hơn
    )
    
    # dọn VRAM
    print("Đang dọn dẹp RAM & VRAM...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost...")
    # train bằng tập train, early stop bằng tập valid
    try:
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    # test trên tập test
    print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm
import torch
import gc
from torch_geometric.loader import NeighborLoader

# tương tự như hàm trên nhưng sẽ trả về ma trận nối giữa raw features và embedding GAT để train XGBoost với cả 2 loại đặc trưng
@torch.no_grad()
def extract_concat_features_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...")
    
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[20, 10],
        input_nodes=mask,
        batch_size=512,
        shuffle=False,
        num_workers=0 
    )
    
    all_combined = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Nối")
    for batch in pbar:
        batch = batch.to(device)
        
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        bs = batch.batch_size
        
        # lấy raw features và embedding các node gốc
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        
        # concat raw_features và embeddings 
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        
        all_combined.append(combined_matrix)
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        del batch, emb, _, raw_features, embeddings, combined_matrix
        
    pbar.close()
    

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    final_combined = np.concatenate(all_combined, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Nối: {final_combined.shape}")
    return final_combined, final_labels

def train_evaluate_concat_xgboost(X_train, y_train, X_valid, y_valid, X_test, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.power(w, 0.8) for c, w in zip(unique_classes, raw_class_weights)}
    
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=8, 
        min_child_weight=2,
        gamma=0.2,
        reg_lambda=1.5,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.8,
        max_delta_step=2,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=100
    )
    
    print(" Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)")
    try:
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang Train trên:", device)

print("=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT ===")
import gc
gc.collect()

num_features = super_graph_faiss.x.shape[1] 

# lấy số classs
num_classes = len(torch.unique(super_graph_faiss.y)) 
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# khởi tạo model GAT
model_faiss = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=128,      
     embedding_dim=64,        
     num_classes=num_classes, 
     heads=8,
     edge_dropout=0.3,
     edge_dim=4 
).to(device)

# tạo mảng Class_Weights dựa trên tập train
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_y_cpu = super_graph_faiss.y[super_graph_faiss.train_mask].cpu().numpy()
unique_classes_arr = np.unique(train_y_cpu)
raw_weights_arr = compute_class_weight('balanced', classes=unique_classes_arr, y=train_y_cpu)

# căn bậc 2 để làm mượt trọng số
smoothed_weights = np.sqrt(raw_weights_arr)
# np.clip: giới hạn giá trị của trọng số trong khoảng [0.1,10.0]
smoothed_weights = np.clip(smoothed_weights, a_min=0.1, a_max=10.0)

# chuyển thành tensor và đưa lên device
gat_class_weights_faiss = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)
print(f"Trọng số cân bằng Class Weights GAT: {gat_class_weights_faiss}")

print("\nBắt đầu quá trình Huấn Luyện GAT!")
# chạy hàm train_gat
model_faiss = train_gat(
    model_faiss, 
    train_loader_faiss, 
    valid_loader_faiss, 
    device, 
    epochs=20, 
    lr=0.0005, 
    class_weights=gat_class_weights_faiss,
    patience=6 
)

import os
os.makedirs("model_final", exist_ok=True)
torch.save(model_faiss.state_dict(), "model_final/gat_embedder.pth")
print("\nThành công! Đã lưu trọng số GAT tĩnh vào thư mục 'model_final/gat_embedder.pth'")

Đang Train trên: cuda
=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT ===


Số lượng Classes (Loại tấn công): 13
Trọng số cân bằng Class Weights GAT: tensor([ 1.0150,  0.6429,  1.6584,  0.6771,  0.9592,  1.0407,  0.6063,  1.9159,
         1.2695, 10.0000, 10.0000,  3.1240,  0.8683], device='cuda:0')

Bắt đầu quá trình Huấn Luyện GAT!


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/20 [Train]:   0%|          | 0/1609 [00:00<?, ?it/s]

Epoch 1/20 [Valid]:   0%|          | 0/2146 [00:00<?, ?it/s]

...

Epoch 13/20 [Valid]:   0%|          | 0/2146 [00:00<?, ?it/s]

=== Epoch 13 Tổng kết ===
   Train Loss: 0.0038 | Train Macro F1: 0.9998
   Valid Loss: 0.0020   | Valid Macro F1: 0.9990
   Learning Rate hiện tại: 0.000125
Báo động Early Stopping: Không cải thiện F1 5/6 epochs.


Epoch 14/20 [Train]:   0%|          | 0/1609 [00:00<?, ?it/s]

Epoch 14/20 [Valid]:   0%|          | 0/2146 [00:00<?, ?it/s]

=== Epoch 14 Tổng kết ===
   Train Loss: 0.0038 | Train Macro F1: 0.9999
   Valid Loss: 0.0024   | Valid Macro F1: 0.9989
   Learning Rate hiện tại: 0.000063
Báo động Early Stopping: Không cải thiện F1 6/6 epochs.

Đã kích hoạt Early Stopping tại Epoch 14! Lùi về checkpoint tốt nhất.

Hoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: 0.9990) để dùng trích xuất Embedding!

Thành công! Đã lưu trọng số GAT tĩnh vào thư mục 'model_final/gat_embedder.pth'


In [9]:
num_features = super_graph_faiss.x.shape[1] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# lấy số classs
num_classes = len(torch.unique(super_graph_faiss.y)) 
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# khởi tạo model GAT
model_faiss = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=128,      
     embedding_dim=64,        
     num_classes=num_classes, 
     heads=8,
     edge_dropout=0.3,
     edge_dim=4 
).to(device)

# load trọng số đã train
model_faiss.load_state_dict(torch.load("model_final/gat_embedder.pth", map_location=device))

Số lượng Classes (Loại tấn công): 13


C:\Users\Admin\AppData\Local\Temp\ipykernel_6600\173910689.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_faiss.load_state_dict(torch.load("model_final/gat_embed

<All keys matched successfully>

In [47]:
# giải phóng RAM
import gc
gc.collect()

print("=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===")
# trích xuất sang Vector cho XGBoost xử lý
print("\nTrích xuất Vector cho Train Mask")
X_train_faiss, y_train_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)

print("\nTrích xuất Vector cho Valid Mask")
X_valid_faiss, y_valid_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)

print("\nTrích xuất Vector cho Test Mask")
X_test_faiss, y_test_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

# huấn luyện và đánh giá XGBoost từ embedding của GAT
hybrid_xgb_faiss = train_evaluate_xgboost(X_train_faiss, y_train_faiss, X_valid_faiss, y_valid_faiss, X_test_faiss, y_test_faiss)

# lưu model xgboost đã huấn luyện
import os
os.makedirs("model_final", exist_ok=True)
hybrid_xgb_faiss.save_model("model_final/GAT_XGB_Hybrid_FAISS_Model.json")
print("\nThành công! Hybrid XGBoost FAISS Model (Base) đã lưu vào 'model_final/GAT_XGB_Hybrid_FAISS_Model.json'")

=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===

Trích xuất Vector cho Train Mask
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/6436 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (3294945, 64)

Trích xuất Vector cho Valid Mask
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/8581 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (4393262, 64)

Trích xuất Vector cho Test Mask
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/6436 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (3294963, 64)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
Đang dọn dẹp RAM & VRAM...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.98707	validation_1-mlogloss:1.99760
[97]	validation_0-mlogloss:0.00064	validation_1-mlogloss:0.19741
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 57

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---
Accuracy:            93.45%
F1-Score (Macro):    91.13%
F1-Score (Weighted): 93.26%

Classification Report:
              precision    recall  f1-score   support

           0     0.5926    0.9843    0.7398    246000
           1     0.9844    0.9987    0.9915    613218
           2     0.9525    0.9031    0.9271     92157
           3     0.9992    1.0000    0.9996    552847
           4     0.9999    0.8909    0.9423    275494
           5     0.9996    1.0000    0.9998    234000
           6     0.9900    1.0000    0.9950    689483
       

In [12]:
# thử GAT + XGB với CONCAT (Raw Features + Embeddings) để xem có cải thiện không
print("\n=== BƯỚC 4: TRÍCH XUẤT VÀ NỐI (CONCAT) FEATURES + EMBEDDINGS CHO HYBRID XGBOOST ===")
X_train_concat, y_train_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)
X_valid_concat, y_valid_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)
X_test_concat, y_test_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device) 
hybrid_xgb_concat = train_evaluate_concat_xgboost(X_train_concat, y_train_concat, X_valid_concat, y_valid_concat, X_test_concat, y_test_concat)

import os
os.makedirs("model_final", exist_ok=True)
hybrid_xgb_concat.save_model("model_final/GAT_XGB_Hybrid_Concat_Model.json")
print("\nThành công! Hybrid XGBoost Concat Model (Raw + Embeddings) đã lưu vào 'model_final/GAT_XGB_Hybrid_Concat_Model.json'")


=== BƯỚC 4: TRÍCH XUẤT VÀ NỐI (CONCAT) FEATURES + EMBEDDINGS CHO HYBRID XGBOOST ===
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/6436 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (3294945, 387)
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/8581 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (4393262, 387)
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/6436 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (3294963, 387)

--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---
Đang tính toán Custom Smoothed Class Weights...
 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)
[0]	validation_0-mlogloss:2.24150	validation_1-mlogloss:2.24166
[100]	validation_0-mlogloss:0.01155	validation_1-mlogloss:0.01294
[200]	validation_0-mlogloss:0.00010	validation_1-mlogloss:0.00082
[300]	validation_0-mlogloss:0.00001	validation_1-mlogloss:0.00056
[400]	validation_0-mlogloss:0.00001	validation_1-mlogloss:0.00055
[500]	validation_0-mlogloss:0.00001	validation_1-mlogloss:0.00055
[592]	validation_0-mlogloss:0.00001	validation_1-mlogloss:0.00055
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 492

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---
Accuracy:            99.45%
F1-Score (Macro):    96.07%
F1-Score (Weighted): 99.48%

Classification Report:
              precision    recall  f1-s

In [13]:
# Dùng XGBoost Raw Features làm baseline để so sánh
print("\n=== BƯỚC 5: ĐÁNH GIÁ BASELINE XGBOOST CHỈ RAW FEATURES (KHÔNG DÙNG EMBEDDING) ===")
# Trích xuất Raw Features thuần túy cho từng Mask (Train/Valid/Test)
def extract_raw_features_mask(graph_data, mask):
    print("Đang trích xuất Raw Features thuần túy...")
    features = graph_data.x[mask].cpu().numpy()
    labels = graph_data.y[mask].cpu().numpy()
    print(f"Trích xuất thành công! Kích thước Ma trận Raw Features: {features.shape}")
    return features, labels
X_train_raw, y_train_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.train_mask)
X_valid_raw, y_valid_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.valid_mask)
X_test_raw, y_test_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.test_mask)
baseline_xgb_raw = train_evaluate_xgboost(X_train_raw, y_train_raw, X_valid_raw, y_valid_raw, X_test_raw, y_test_raw)



=== BƯỚC 5: ĐÁNH GIÁ BASELINE XGBOOST CHỈ RAW FEATURES (KHÔNG DÙNG EMBEDDING) ===
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (3294945, 323)
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (4393262, 323)
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (3294963, 323)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
Đang dọn dẹp RAM & VRAM...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.98749	validation_1-mlogloss:1.98810
[100]	validation_0-mlogloss:0.00176	validation_1-mlogloss:0.00231
[200]	validation_0-mlogloss:0.00087	validation_1-mlogloss:0.00152
[251]	validation_0-mlogloss:0.00080	validation_1-mlogloss:0.00152
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 211

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---
Accuracy:            99.29%
F1-Score (Macro):    95.80%
F1-S

In [14]:
# đánh giá GAT trực tiếp trên tập Test Mask (Không qua XGBoost)
@torch.no_grad()
def evaluate_gat_directly(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...")
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[20, 10],
        input_nodes=mask,
        batch_size=512, 
        shuffle=False,
        num_workers=0 
    )
    all_preds = []
    all_labels = []
    pbar = tqdm(loader, desc="Đang đánh giá trực tiếp GAT")
    for batch in pbar:  
        batch = batch.to(device)
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
        bs = batch.batch_size
        all_preds.append(out[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        del batch, out
    pbar.close()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    final_preds = np.concatenate(all_preds, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)   
    pred_classes = final_preds.argmax(axis=1)
    acc = accuracy_score(final_labels, pred_classes)
    f1_macro = f1_score(final_labels, pred_classes, average='macro')
    f1_weighted = f1_score(final_labels, pred_classes, average='weighted')
    print(f"GAT Đánh giá trực tiếp trên Test Mask:")
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(final_labels, pred_classes, digits=4))

# Giải phóng VRAM trước khi chạy đánh giá
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

evaluate_gat_directly(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...


Đang đánh giá trực tiếp GAT:   0%|          | 0/6436 [00:00<?, ?it/s]

GAT Đánh giá trực tiếp trên Test Mask:
Accuracy:            98.86%
F1-Score (Macro):    95.42%
F1-Score (Weighted): 98.89%

Classification Report:
              precision    recall  f1-score   support

           0     0.9994    0.9612    0.9799    246000
           1     0.9772    1.0000    0.9884    613218
           2     0.9594    0.8745    0.9150     92157
           3     1.0000    0.9999    1.0000    552847
           4     0.9999    0.9572    0.9781    275494
           5     1.0000    1.0000    1.0000    234000
           6     1.0000    1.0000    1.0000    689483
           7     1.0000    0.9418    0.9700     69052
           8     0.9688    1.0000    0.9841    157261
           9     0.9905    0.8461    0.9126      1962
          10     0.8183    1.0000    0.9001      1360
          11     0.6405    0.9866    0.7767     25973
          12     1.0000    0.9999    0.9999    336156

    accuracy                         0.9886   3294963
   macro avg     0.9503    0.9667    0.95

In [57]:
def train_evaluate_concat_xgboost(X_train, y_train, X_test, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.power(w, 0.8) for c, w in zip(unique_classes, raw_class_weights)}
    
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=8, 
        min_child_weight=2,
        gamma=0.2,
        reg_lambda=1.5,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.8,
        max_delta_step=2,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
    )
    
    print(" Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)")
    try:
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            verbose=100 
        )
        

    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [58]:
train_valid = train_evaluate_concat_xgboost(X_valid_concat, y_valid_concat, X_test_concat, y_test_concat)


--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---
Đang tính toán Custom Smoothed Class Weights...
 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---
Accuracy:            98.24%
F1-Score (Macro):    95.23%
F1-Score (Weighted): 98.23%

Classification Report:
              precision    recall  f1-score   support

           0     0.9965    1.0000    0.9982    246000
           1     0.9847    0.9999    0.9922    613218
           2     0.9729    0.8878    0.9284     92157
           3     0.9722    1.0000    0.9859    552847
           4     1.0000    0.8451    0.9161    275494
           5     0.9995    1.0000    0.9998    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9943    0.9409    0.9668     69052
           8     0.8977    1.0000    0.9461    157261
           9     1.0000    0.8496    0.9187      1962
          10     0.8287    0